# Data Preprocessing

## Business Objective

Machine learning models require data to be converted into a numerical format and organized appropriately before training.

This notebook prepares the engineered dataset by:

- Encoding categorical variables
- Selecting input features
- Splitting the data chronologically
- Scaling numerical features
- Preparing sequential data for the LSTM model

In [1]:
# ==========================================================
# Data Preprocessing
# ==========================================================

import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import QuantileTransformer

import joblib

pd.set_option("display.max_columns", None)

import warnings
warnings.filterwarnings("ignore")

In [2]:
# ==========================================================
# Load Engineered Dataset
# ==========================================================

df = pd.read_csv("../data/burger_data_engineered.csv")

df["Date"] = pd.to_datetime(df["Date"])

print(df.shape)

df.head()

(24422, 21)


,Date,Region,Temperature,Humidity,Wind,Visibility,Pressure,Sales,Year,Month,Day,DayOfWeek,Quarter,WeekOfYear,IsWeekend,Sales_Lag_1,Sales_Lag_7,Sales_Lag_30,Rolling_Mean_7,Rolling_Mean_30,Rolling_STD_7
0,2014-01-03,Reg2,18.174600,0.435980,24.228756,13.744171,1016.652700,992.37,2014,1,3,4,1,1,0,204.32,699.35,1539.45,1581.357143,1639.600333,1291.820530
1,2014-01-04,Reg10,21.861622,0.402545,11.281557,4.461844,1023.733652,23.83,2014,1,4,5,1,1,1,992.37,2774.98,2721.81,1623.217143,1621.364333,1262.900440
2,2014-01-04,Reg8,3.776241,0.886265,5.661404,9.982186,1020.505489,672.21,2014,1,4,5,1,1,1,23.83,950.34,2240.22,1230.195714,1531.431667,1272.774252
3,2014-01-04,Reg3,0.196468,0.829441,5.494188,8.846484,1024.717837,664.99,2014,1,4,5,1,1,1,672.21,405.83,468.78,1190.462857,1479.164667,1287.225940
4,2014-01-04,Reg9,23.463183,0.724208,10.477588,11.057665,1010.657532,2765.08,2014,1,4,5,1,1,1,664.99,3087.19,137.76,1227.485714,1485.705000,1264.422258


In [3]:
import os

# ==========================================================
# Encode Region
# ==========================================================

encoder = LabelEncoder()

df["Region"] = encoder.fit_transform(df["Region"])

print("Encoded Regions:")

print(df["Region"].unique())

os.makedirs("../models", exist_ok=True)
joblib.dump(encoder, "../models/region_encoder.pkl")


Encoded Regions:
[2 1 8 3 9 6 7 4 5 0]


['../models/region_encoder.pkl']

In [4]:
# ==========================================================
# Feature Selection
# ==========================================================

X = df.drop(
    columns=[
        "Sales",
        "Date"
    ]
)

y = df["Sales"]

print(X.shape)

print(y.shape)

X.head()

(24422, 19)
(24422,)


,Region,Temperature,Humidity,Wind,Visibility,Pressure,Year,Month,Day,DayOfWeek,Quarter,WeekOfYear,IsWeekend,Sales_Lag_1,Sales_Lag_7,Sales_Lag_30,Rolling_Mean_7,Rolling_Mean_30,Rolling_STD_7
0,2,18.174600,0.435980,24.228756,13.744171,1016.652700,2014,1,3,4,1,1,0,204.32,699.35,1539.45,1581.357143,1639.600333,1291.820530
1,1,21.861622,0.402545,11.281557,4.461844,1023.733652,2014,1,4,5,1,1,1,992.37,2774.98,2721.81,1623.217143,1621.364333,1262.900440
2,8,3.776241,0.886265,5.661404,9.982186,1020.505489,2014,1,4,5,1,1,1,23.83,950.34,2240.22,1230.195714,1531.431667,1272.774252
3,3,0.196468,0.829441,5.494188,8.846484,1024.717837,2014,1,4,5,1,1,1,672.21,405.83,468.78,1190.462857,1479.164667,1287.225940
4,9,23.463183,0.724208,10.477588,11.057665,1010.657532,2014,1,4,5,1,1,1,664.99,3087.19,137.76,1227.485714,1485.705000,1264.422258


In [5]:
# ==========================================================
# Time-Based Split
# ==========================================================

train_size = int(len(df) * 0.70)

validation_size = int(len(df) * 0.15)

X_train = X.iloc[:train_size]

X_validation = X.iloc[
    train_size:
    train_size + validation_size
]

X_test = X.iloc[
    train_size + validation_size:
]

y_train = y.iloc[:train_size]

y_validation = y.iloc[
    train_size:
    train_size + validation_size
]

y_test = y.iloc[
    train_size + validation_size:
]

print("Training :", X_train.shape)

print("Validation :", X_validation.shape)

print("Testing :", X_test.shape)

Training : (17095, 19)
Validation : (3663, 19)
Testing : (3664, 19)


In [6]:
# ==========================================================
# Feature Scaling
# ==========================================================

scaler = QuantileTransformer(
    output_distribution="normal",
    random_state=42
)

# Fit only on training data
X_train_scaled = scaler.fit_transform(X_train)

# Transform validation and test data
X_validation_scaled = scaler.transform(X_validation)

X_test_scaled = scaler.transform(X_test)

print("Training shape:", X_train_scaled.shape)
print("Validation shape:", X_validation_scaled.shape)
print("Testing shape:", X_test_scaled.shape)

Training shape: (17095, 19)
Validation shape: (3663, 19)
Testing shape: (3664, 19)


In [7]:
# ==========================================================
# Save Scaler
# ==========================================================

joblib.dump(
    scaler,
    "../models/quantile_scaler.pkl"
)

print("Scaler saved successfully.")

Scaler saved successfully.


In [8]:
# ==========================================================
# Function to Create LSTM Sequences
# ==========================================================

def create_sequences(features, target, sequence_length=30):

    X_seq = []
    y_seq = []

    for i in range(sequence_length, len(features)):
        X_seq.append(features[i-sequence_length:i])
        y_seq.append(target.iloc[i])

    return np.array(X_seq), np.array(y_seq)

In [9]:
# ==========================================================
# Generate LSTM Sequences
# ==========================================================

sequence_length = 30

X_train_seq, y_train_seq = create_sequences(
    X_train_scaled,
    y_train,
    sequence_length
)

X_validation_seq, y_validation_seq = create_sequences(
    X_validation_scaled,
    y_validation,
    sequence_length
)

X_test_seq, y_test_seq = create_sequences(
    X_test_scaled,
    y_test,
    sequence_length
)

print("Training sequences:", X_train_seq.shape)
print("Validation sequences:", X_validation_seq.shape)
print("Testing sequences:", X_test_seq.shape)

Training sequences: (17065, 30, 19)
Validation sequences: (3633, 30, 19)
Testing sequences: (3634, 30, 19)


In [10]:
# ==========================================================
# Save Processed Data
# ==========================================================

np.save("../data/X_train_seq.npy", X_train_seq)
np.save("../data/X_validation_seq.npy", X_validation_seq)
np.save("../data/X_test_seq.npy", X_test_seq)

np.save("../data/y_train_seq.npy", y_train_seq)
np.save("../data/y_validation_seq.npy", y_validation_seq)
np.save("../data/y_test_seq.npy", y_test_seq)

print("Processed LSTM datasets saved successfully.")

Processed LSTM datasets saved successfully.


# Preprocessing Summary

The preprocessing pipeline included:

- Encoding the categorical region feature.
- Separating input features and target values.
- Performing a chronological train, validation, and test split.
- Scaling numerical features using a QuantileTransformer fitted only on the training data.
- Creating 30-day sequences for the LSTM model.
- Saving the preprocessing objects and processed datasets for reproducibility.

In [11]:
# ==========================================================
# Save Final Processed Dataset
# ==========================================================

# Save the fully processed dataset
df.to_csv(
    "../data/burger_data_processed.csv",
    index=False
)

print("Processed dataset saved successfully!")

print("Dataset Shape:", df.shape)
print(df.shape)

print(df.columns)

Processed dataset saved successfully!
Dataset Shape: (24422, 21)
(24422, 21)
Index(['Date', 'Region', 'Temperature', 'Humidity', 'Wind', 'Visibility',
       'Pressure', 'Sales', 'Year', 'Month', 'Day', 'DayOfWeek', 'Quarter',
       'WeekOfYear', 'IsWeekend', 'Sales_Lag_1', 'Sales_Lag_7', 'Sales_Lag_30',
       'Rolling_Mean_7', 'Rolling_Mean_30', 'Rolling_STD_7'],
      dtype='object')
